In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement
#%run ../delta_function ----- A decommenter pour lancer le notebook separement
#%run ./delta_schema ----- A decommenter pour lancer le notebook separement
#%run ./load_data ----- A decommenter pour lancer le notebook separement
#%run ./dim_batch ----- A decommenter pour lancer le notebook separement
#%run ./dim_parameter ----- A decommenter pour lancer le notebook separement

In [0]:
#%run ./env

In [0]:
#%run ./python_libraries 

In [0]:
#%run ../delta_function 

In [0]:
#%run ./delta_schema 

In [0]:
#%run ./load_data 

In [0]:
#%run ./transform_data 

In [0]:
# Pas de filtre sur `deleted` ici, meme raison que dans dim_batch : le mode
# update de handle_table_update ne fait que des inserts et des updates, jamais
# de delete. Une mesure ecartee a la source disparait du snapshot, sa ligne
# cible n'est plus jamais revisitee par le MERGE et resterait figee avec
# deleted = False. Sur un fait c'est plus grave que sur une dimension : la
# valeur n'est pas seulement affichee, elle est SOMMEE pour toujours dans les
# KPI energie, sans aucun signal. En laissant passer les supprimees, `deleted`
# fait partie des additional_columns_to_check et bascule a True au run suivant.
# Le filtre deleted = False se fait cote Power BI.
measures_automatic = (
    batches_production_monitoring
    .select(
        F.col("id_batch_production_monitoring").alias("measure_id"),
        F.col("batch").alias("batch_id"),
        F.col("parameter").alias("parameter_id"),
        F.col("event_date"),
        F.col("value"),
        F.col("production_cell"),
        F.col("deleted"),
        F.lit("automatic").alias("source"),
        F.col("created_at")
    )
)

# Dedup defensive : aucun doublon detecte a ce jour sur (batch_id, parameter_id, event_date),
# mais on applique la meme regle que sur manual_entries par securite (garder le plus recent).
#
# `deleted` en premier critere de tri depuis qu'on laisse passer les lignes
# supprimees : sans ca, une ligne supprimee plus recente que la ligne vivante
# du meme triplet gagnerait le rn = 1 et ferait disparaitre la vivante du
# snapshot -- qui ne serait alors plus jamais mise a jour par le MERGE.
# Le vivant prime donc toujours ; une ligne supprimee ne remonte que si elle
# n'a pas de concurrente vivante, ce qui est precisement le cas a propager.
window_dedup_automatic = (
    Window.partitionBy("batch_id", "parameter_id", "event_date")
    .orderBy(F.col("deleted").asc(), F.col("created_at").desc())
)

measures_automatic = (
    measures_automatic
    .withColumn("rn", F.row_number().over(window_dedup_automatic))
    .filter(F.col("rn") == 1)
    .drop("rn", "created_at")
)

# On ne garde que les mesures automatiques qui ont une correspondance dans
# energy_metadata (inner join via parameters_variables.code = measure_name).
# rapprochement sur le code uniquement (pas de filtre par prd_line).
energy_metadata_codes = (
    parameters_variables.alias("pv")
    .join(
        energy_metadata.alias("em"),
        F.col("pv.code") == F.col("em.measure_name"),
        "inner"
    )
    .select(F.col("pv.id_parameter_variable").alias("parameter_id"))
    .distinct()
)

measures_automatic = (
    measures_automatic.join(
        F.broadcast(energy_metadata_codes),
        on="parameter_id",
        how="inner"
    )

    .select(
        "measure_id",
        "batch_id",
        "parameter_id",
        "event_date",
        "value",
        "production_cell",
        "deleted",
        "source"
    )
)

measures_manual = (
    manual_entries
    .select(
        F.col("id_manual_entry").alias("measure_id"),
        F.col("batch").alias("batch_id"),
        F.col("parameter").alias("parameter_id"),
        F.col("entry_date").alias("event_date"),
        F.col("value"),
        F.lit(None).cast("int").alias("production_cell"),
        F.col("deleted"),
        F.lit("manual").alias("source"),
        F.col("created_at")
    )
)

# On ne garde que les 3 mesures manuelles legitimes pour ce rapport
# (regle metier confirmee), via leur code (stable), pas leur ID.
# CONSEQUENCE ASSUMEE : toute autre saisie manuelle presente dans la
# source (ex: contournements ponctuels d'un trou dans energy_metadata)
# est exclue, meme si ca laisse des colonnes vides en attendant que
# energy_metadata soit corrigee cote source. C'est le comportement
# voulu, pas un bug : ces saisies ne sont pas une source autorisee.
codes_manuels_attendus = ["malt_weight", "goods_weight", "malt_yield_r2"]

parameter_ids_manuels_attendus = (
    parameters_variables
    .filter(F.col("code").isin(codes_manuels_attendus))
    .select(F.col("id_parameter_variable").alias("parameter_id"))
)

measures_manual = (
    measures_manual.join(
        F.broadcast(parameter_ids_manuels_attendus),
        on="parameter_id",
        how="inner"
    )
    .select(
        "measure_id", "batch_id", "parameter_id", "event_date",
        "value", "production_cell", "deleted", "source", "created_at"
    )
)

# Dedup : doublons detectes sur (batch_id, parameter_id, event_date) -
# certains sont de vrais doublons techniques (meme valeur, meme created_at),
# d'autres sont des corrections (valeur differente, created_at different).
# On garde dans les 2 cas la ligne la plus recente par created_at.
# `deleted` en premier critere de tri : meme raison que pour window_dedup_automatic.
window_dedup_manual = (
    Window.partitionBy("batch_id", "parameter_id", "event_date")
    .orderBy(F.col("deleted").asc(), F.col("created_at").desc())
)

measures_manual = (
    measures_manual
    .withColumn("rn", F.row_number().over(window_dedup_manual))
    .filter(F.col("rn") == 1)
    .drop("rn", "created_at")
)

# Union via SQL (unionByName appelle _jdf, non supporte sur cluster partage Unity Catalog)
measures_automatic.createOrReplaceTempView("measures_automatic")
measures_manual.createOrReplaceTempView("measures_manual")

union_columns = "measure_id, batch_id, parameter_id, event_date, value, production_cell, deleted, source"

fact_batch_measures = spark.sql(f"""
    SELECT {union_columns} FROM measures_automatic
    UNION ALL
    SELECT {union_columns} FROM measures_manual
""")

## Ecriture Delta

In [0]:
current_process = "fact_batch_measures"
target_fact_batch_measures = current_catalog + "." + current_schema + "." + current_process
print(target_fact_batch_measures)

In [0]:
all_columns = fact_batch_measures.columns
primary_key = ['measure_id']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

In [0]:
align_target_schema(
    fact_batch_measures,
    target_fact_batch_measures,
    verbose=(verbose_mode == 'debug')
)

handle_table_update(
    fact_batch_measures,
    target_fact_batch_measures,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode
)